# Extrapolate end-to-end (intelligrate.extrapolate)

This notebook runs the full `intelligrate.extrapolate` workflow using the **Python API**:
1) nested CV training (OOF outputs)
2) full fit (final model artifact)
3) full predict (new samples + optional evaluation on paired subset)

Outputs are written to `results/`.

It uses the example data in `data/`.


## Install (pip)
If you are running this notebook outside the repo, install with pip.



In [6]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [7]:
pwd

'/Users/meyeanni/Desktop/git_sourdough/Intelligrate/docs/notebooks'

install intelligrate (if not yet installed):


In [ ]:
# Uncomment if needed:
# !pip install "intelligrate @ git+https://github.com/ORG/REPO.git@vX.Y.Z"



In [8]:
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists():
            return p
    raise FileNotFoundError('Could not find repo root (pyproject.toml)')

repo_root = find_repo_root(Path.cwd())

# Ensure local package imports work without installation
sys.path.insert(0, str(repo_root / 'src'))

results_dir = repo_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

print('Repo root:', repo_root)
print('Results dir:', results_dir)


Repo root: /Users/meyeanni/Desktop/git_sourdough/Intelligrate
Results dir: /Users/meyeanni/Desktop/git_sourdough/Intelligrate/results


## Load inputs
- `X_kmers.tsv`: paired samples (k-mers)
- `X_kmers_full.tsv`: all samples (paired + unpaired, for extrapolation)
- `Y_kos.tsv`: KO profiles for paired samples
- `ko_to_superclass.tsv`: KO -> pathway/superclass mapping, optional


In [9]:
X_full = pd.read_csv(repo_root / 'data' / 'X_kmers_full.tsv', sep='	', index_col=0)
X = pd.read_csv(repo_root / 'data' / 'X_kmers.tsv', sep='	', index_col=0)
Y = pd.read_csv(repo_root / 'data' / 'Y_kos.tsv', sep='	', index_col=0)

print('X_full:', X_full.shape)
print('X:', X.shape)
print('Y:', Y.shape)


X_full: (610, 12340)
X: (95, 12340)
Y: (95, 4211)


### Optional: Pre-filter Y before any modeling
If you want to pre-filter KO features (e.g., apply a detection threshold once globally and keep zeroes as informative), do it here **once** and then use `Y` everywhere downstream.
This avoids per-fold KO dropping, but it changes the modeling assumptions.

**Important:** if you do this, pass the filtered `Y` into all downstream calls (training, fixed-param sweep, evaluation, PICRUSt2 comparisons).


In [ ]:
# --- Optional global Y pre-filtering (edit as needed) ---
# Example: apply a detect threshold once and keep zeros as zeros.
# This is global pre-filtering; be aware this uses all samples (no CV isolation).
# If you enable this, replace Y with Y_prefiltered and use it everywhere downstream.
#
# detect_threshold = model_cfg['y_detect_threshold']
# Y_prefiltered = Y.copy()
# Y_prefiltered = Y_prefiltered.mask(Y_prefiltered < detect_threshold, 0.0)
# Y = Y_prefiltered

# NOTE: After this point, make sure any call uses the updated Y:
# - train/_run_once or nested CV inputs
# - fixed_param_oof_knn_on_embedding
# - evaluate_paired_subset
# - PICRUSt2 comparisons (align to the same KO set as Y)


## Parameters
Parameters that we define here correspond to what can be found in the **defaults** as `configs/default.yaml`, but now we expose them here in the notebook for easier editing.


If you want to see every parameter and full explanations, also check:
- `docs/TUTORIAL_extrapolate.md`
- `src/intelligrate/extrapolate/train.py` and `src/intelligrate/extrapolate/cv_knn.py`


In [20]:
# ---- Data file names (used by the training API) ----
data_cfg = {
    'x_full': 'X_kmers_full.tsv',
    'x': 'X_kmers.tsv',
    'y': 'Y_kos.tsv',
    'picrust2': 'picrust2_kos.tsv', # optional for comparison
    'ko_to_superclass': 'ko_to_superclass.tsv', # optional for pathway RMSE
}

# ---- Cross-validation settings ----
cv_cfg = {
    # outer_splits/inner_splits: more splits = slower but more stable estimates
    'outer_splits': 5,
    'inner_splits': 3,
    'seed': 0,
    # informed_splits: use metadata to form splits (False by default)
    'informed_splits': False,
}

# ---- X embedding (k-mer -> low-dim) ----
embed_cfg = {
    # min_prev_x_abs: drop k-mers present in fewer samples than this
    'min_prev_x_abs': 30,
    # pseudocount_x: added before CLR
    'pseudocount_x': 0.5,
    # n_components: SVD dimensions for the k-mer embedding (X space)
    'n_components': 128,
}

# ---- Model / prediction settings ----
model_cfg = {
    # min_prev_y_abs: drop KOs seen in fewer samples than this
    'min_prev_y_abs': 1,
    # y_detect_threshold: detection threshold in TSS space
    'y_detect_threshold': 1000.0,
    # pseudocount_y: added before CLR
    'pseudocount_y': 0.5 / 1e6,

    # KNN hyperparameters (grid for nested CV)
    'neigh_k_grid': [20, 24, 28, 32],
    'tau_mult_grid': [0.5, 1.0, 2.0, 4.0],
    'lam_grid': [0.0],
    'y_latent_k_grid': [0, 10, 20],

    # Metric learning
    'use_metric_learning': True,
    'metric_max_pairs': 5000,
    'metric_ridge_grid': [1.0, 2.5, 5.0],

    # Out-of-distribution shrinkage
    'ood_shrink': True,
    'ood_shrink_inner': True,
    'ood_lam_base': 0.7,
    'ood_lam_cap': 0.5,
    'ood_tau_inflate': False,
}

# ---- Objective weights ----
objective_cfg = {
    # w_dm is the primary objective (Aitchison DM Spearman)
    'w_dm': 1.0,
    'w_wclr': 0.0,
    'w_pw_rmse': 0.0,
    'w_softf1': 0.0,
    'w_jsd': 0.0,
}

# ---- Precision/recall settings ----
prf_cfg = {
    'prf_thresh': 1.0e-6,
    'prf_weight': 'binary', 
}

# ---- Optional metrics ----
metrics_cfg = {
    'compute_wclr': True,
    'compute_jsd': True,
    'compute_pathway_rmse': True,
    'pathway_rmse_per_group': True,
    'pathway_rmse_log1p': True,
}

# ---- Score settings (for the optional global OOF DM check) ----
score_cfg = {
    'min_prev_y_abs': 1,
    'y_detect_threshold': 3000.0,
    'pseudocount_y': 0.5 / 1e6,
}


## 1) Train on paired-samples (nested CV)
We call the training API directly and then write the outputs to `results/`.

This step is to get out-of-fold (OOF) predictions for the paired samples, which can be used for initial model evaluation and calibration/hyperparameter tuning and selection.


In [21]:
from intelligrate.extrapolate import train as train_model

cfg = {
    'data': data_cfg,
    'cv': cv_cfg,
    'embed': embed_cfg,
    'model': model_cfg,
    'objective': objective_cfg,
    'prf': prf_cfg,
    'metrics': metrics_cfg,
    'score': score_cfg,
}

payload = train_model._run_once(cfg, data_dir=repo_root / 'data', out_dir=results_dir)

# Write outputs (same files as the CLI)
oof_clr = payload['oof_clr']
oof_tss = payload['oof_tss']
folds = payload['folds']
run = payload['run']

# Save core outputs
(oof_clr).to_csv(results_dir / 'oof_clr.tsv', sep='	')
(oof_tss).to_csv(results_dir / 'oof_tss.tsv', sep='	')
folds.to_csv(results_dir / 'folds.tsv', sep='	', index=False)
(results_dir / 'summary.json').write_text(json.dumps(run, indent=2))

summary_flat = {k: v for k, v in run.items() if k != 'config'}
pd.DataFrame([summary_flat]).to_csv(results_dir / 'summary.tsv', sep='	', index=False)

print('OBJECTIVE_DM_SPEARMAN_MEAN:', run['objective_dm_spearman_mean'])
print('MODEL_DM_UNION_STRICT:', run['model_dm_union_strict'])


OBJECTIVE_DM_SPEARMAN_MEAN: 0.4064463484292126
MODEL_DM_UNION_STRICT: 0.5229670698671964


In [23]:
summary = json.loads((results_dir / 'summary.json').read_text())
summary_keys = [
    'objective_dm_spearman_mean',
    'model_dm_union_strict',
    'picrust2_dm_union_strict',
    'delta_union',
    'runtime_sec',
]

pd.DataFrame([{k: summary.get(k) for k in summary_keys}])


,objective_dm_spearman_mean,model_dm_union_strict,picrust2_dm_union_strict,delta_union,runtime_sec
0,0.406446,0.522967,0.452675,0.070292,16.772704


  - objective_dm_spearman_mean: primary training objective, averaged Spearman correlation between true vs. predicted
  Aitchison distance matrices across CV folds (higher = better recovery of sample–sample structure).
  - model_dm_union_strict: KO‑union Spearman score on OOF predictions (union of KOs in truth and prediction, computed in CLR/
  Aitchison space; higher = better overall agreement).
  - picrust2_dm_union_strict: same KO‑union score for PICRUSt2 (baseline), optional (only if PICRUSt2 predictions are provided).
  - delta_union: model_dm_union_strict − picrust2_dm_union_strict (positive means the model beats PICRUSt2 in terms of matrix spearman correlations on the KO union).
  - runtime_sec: total training runtime in seconds.

In [24]:
folds = pd.read_csv(results_dir / 'folds.tsv', sep='	')
folds.head()

,fold,split_mode,kmeans_k_used,best_inner_comp,best_inner_dm,dm_spearman,wclr_mse,pw_rmse_log1p,soft_precision,soft_recall,...,shrink_lam_q90,std_dm_true,std_dm_pred,y_keep_n,neigh_k,tau_mult,lam,y_latent_k,metric_ridge,inner_dm
0,1,kfold,NaN,0.188882,0.286622,0.482394,1.075229,0.007383,0.830551,0.965629,...,0.5,29.569735,9.544883,4211,20,1.0,0.0,10,1.0,0.286622
1,2,kfold,NaN,0.254585,0.299358,0.535513,1.175082,0.006372,0.838951,0.979863,...,0.5,27.355864,6.156110,4211,32,1.0,0.0,0,1.0,0.299358
2,3,kfold,NaN,0.306537,0.393309,0.283347,1.108813,0.007034,0.834426,0.992492,...,0.5,18.554112,7.597809,4211,32,0.5,0.0,0,1.0,0.393309
3,4,kfold,NaN,0.193331,0.343167,0.585031,0.893126,0.006689,0.856311,0.986569,...,0.5,23.182926,10.195309,4211,32,0.5,0.0,10,1.0,0.343167
4,5,kfold,NaN,0.340296,0.460748,0.145948,1.696055,0.009137,0.860472,0.954186,...,0.5,29.221874,9.701685,4211,32,0.5,0.0,0,5.0,0.460748


-->> this contains all the 'best' parameter selections for paired sample predictions, which can be considered for evaluation and calibration/hyperparameter tuning and selection.


In [25]:
#how many rows in oof_tss contain any nans?:
oof_tss.isna().any(axis=1).sum()

np.int64(0)

In [15]:
# from intelligrate.extrapolate.metrics import _pairwise_union_mats_tss
# from intelligrate.extrapolate.transforms import clr_rows

# def count_rows_dropped_for_union_strict(Y_truth, oof_tss, pseudocount, detect_threshold):
#       # This matches model_dm_union_strict: union + fillna_zero=True
#       truth_tss_u, pred_tss_u = _pairwise_union_mats_tss(
#           Y_truth, oof_tss, detect_threshold=detect_threshold, fillna_zero=True
#       )
#       truth_clr = clr_rows(truth_tss_u, pseudocount=pseudocount)
#       pred_clr = clr_rows(pred_tss_u, pseudocount=pseudocount)

#       n_total = truth_clr.shape[0]
#       good = truth_clr.notna().all(axis=1) & pred_clr.notna().all(axis=1)
#       n_good = int(good.sum())
#       n_drop = n_total - n_good

#       return {
#           "n_total": n_total,
#           "n_good": n_good,
#           "n_dropped": n_drop,
#           "frac_dropped": (n_drop / n_total if n_total else None),
#       }

#   # Example usage in your notebook:
# count_rows_dropped_for_union_strict(Y, oof_tss, pseudocount=score_cfg['pseudocount_y'], detect_threshold=model_cfg['y_detect_threshold'])

{'n_total': 95, 'n_good': 95, 'n_dropped': 0, 'frac_dropped': 0.0}

### Optional: fixed-parameter sweep for stable hyperparameters

Use this to find a **single fixed hyperparameter set** that performs well for leakage-free OOF.
It evaluates each combo with fixed-parameter OOF and ranks by `dm_union_strict`.

**Important:** if a parameter is *not* listed in `fixed_param_sweep`, the sweep will use the value
from config. For parameters with a `*_grid` (e.g., `neigh_k_grid`), it will use that grid list.
To force a single value, list it explicitly in `fixed_param_sweep`.


In [26]:
#make a dictionary out of ko to superclass mapping:
ko_to_superclass_path = repo_root / 'data' / 'ko_to_superclass.tsv'
ko_to_superclass_df = pd.read_csv(ko_to_superclass_path, sep='	', index_col=0)
ko_to_superclass = ko_to_superclass_df['superclass'].to_dict()
ko_to_superclass

{'K00001': 'SCFA fermentation',
 'K00002': 'Cofactors/vitamins',
 'K00003': 'Energy/central carbon',
 'K00004': 'Energy/central carbon',
 'K00005': 'Energy/central carbon',
 'K00008': 'Energy/central carbon',
 'K00009': 'Energy/central carbon',
 'K00010': 'Energy/central carbon',
 'K00012': 'Carbohydrate metabolism',
 'K00013': 'Energy/central carbon',
 'K00014': 'Tryptophan metabolism',
 'K00015': 'Energy/central carbon',
 'K00016': 'SCFA fermentation',
 'K00018': 'Energy/central carbon',
 'K00019': 'Other',
 'K00020': 'Energy/central carbon',
 'K00024': 'Energy/central carbon',
 'K00026': 'Energy/central carbon',
 'K00027': 'Energy/central carbon',
 'K00029': 'Energy/central carbon',
 'K00030': 'Cofactors/vitamins',
 'K00031': 'Other',
 'K00032': 'Energy/central carbon',
 'K00033': 'Energy/central carbon',
 'K00034': 'Energy/central carbon',
 'K00036': 'Energy/central carbon',
 'K00038': 'Energy/central carbon',
 'K00040': 'Energy/central carbon',
 'K00041': 'Energy/central carbon',


In [27]:
embed_cfg

{'min_prev_x_abs': 30, 'pseudocount_x': 0.5, 'n_components': 128}

In [28]:
from intelligrate.extrapolate.embedding import fit_x_embedding_svd_clr
from intelligrate.extrapolate.fixed_param_sweep import run_fixed_param_sweep_explicit

# Fit embedding on all samples (X_full)
embed = fit_x_embedding_svd_clr(
    X_full,
    min_prev_x_abs=int(embed_cfg['min_prev_x_abs']),
    pseudocount_x=float(embed_cfg['pseudocount_x']),
    n_components=int(embed_cfg['n_components']),
    seed=int(cv_cfg['seed']),
)

embed_path = results_dir / 'embed.joblib'
joblib.dump(embed, embed_path)
print('Saved embedding to:', embed_path)

# Define a small fixed-parameter sweep (uses the same X_full/X/Y/ko_to_superclass as above)
cfg["fixed_param_sweep"] = {
    "neigh_k": [24, 28, 32],
    "tau_mult": [0.5, 1.0],
    "y_latent_k": [10, 20],
    "metric_ridge": [1.0, 2.5],
}

sweep_out = results_dir / "fixed_param_sweep.tsv"
sweep_df = run_fixed_param_sweep_explicit(
    X_full=X_full,
    X=X,
    Y=Y,
    ko_to_superclass=ko_to_superclass,
    out_path=sweep_out,
    cv_cfg=cv_cfg,
    embed_cfg=embed_cfg,
    model_cfg=model_cfg,
    prf_cfg=prf_cfg,
    metrics_cfg=metrics_cfg,
    sweep_cfg=cfg["fixed_param_sweep"],
    embed=embed,
)
sweep_df.head()


Saved embedding to: /Users/meyeanni/Desktop/git_sourdough/Intelligrate/results/embed.joblib


,neigh_k,tau_mult,y_latent_k,metric_ridge,lam,min_prev_y_abs,y_detect_threshold,pseudocount_y,metric_max_pairs,tau_scale_k_nn,...,ood_lam_cap,ood_tau_inflate,ood_tau_gamma,use_metric_learning,outer_splits,seed,informed_splits,dm_union_strict,dm_union_raw,runtime_s
18,32,0.5,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.522370,0.435635,2.429053
19,32,0.5,20,2.5,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.522370,0.435636,1.799002
17,32,0.5,10,2.5,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.519784,0.432483,1.797031
16,32,0.5,10,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.519784,0.432483,1.800107
11,28,0.5,20,2.5,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.516002,0.430341,1.801503


In [29]:
#check which sweep params give the best dm_union_strict:
sweep_df.sort_values('dm_union_strict', ascending=False).head()

,neigh_k,tau_mult,y_latent_k,metric_ridge,lam,min_prev_y_abs,y_detect_threshold,pseudocount_y,metric_max_pairs,tau_scale_k_nn,...,ood_lam_cap,ood_tau_inflate,ood_tau_gamma,use_metric_learning,outer_splits,seed,informed_splits,dm_union_strict,dm_union_raw,runtime_s
18,32,0.5,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.522370,0.435635,2.429053
19,32,0.5,20,2.5,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.522370,0.435636,1.799002
17,32,0.5,10,2.5,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.519784,0.432483,1.797031
16,32,0.5,10,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.519784,0.432483,1.800107
11,28,0.5,20,2.5,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.516002,0.430341,1.801503


-> in this case it is neight_k = 28, tau_mult = 0.5, y_latent_k = 10, metric_ridge = 1.0

### now if you want to try a few more fixed parameter combos, go ahead, otherwise skip and continue with the full fit of the final model further below

In [31]:
# Define a small fixed-parameter sweep (uses the same X_full/X/Y/ko_to_superclass as above), round 2:
cfg["fixed_param_sweep"] = {
    "neigh_k": [32, 34, 36, 38, 40],
    "tau_mult": [0.25, 0.35, 0.5, 0.65, 0.8],
    "y_latent_k": [20],
    "metric_ridge": [1.0],
}

sweep_out2 = results_dir / "fixed_param_sweep2.tsv"
sweep_df2 = run_fixed_param_sweep_explicit(
    X_full=X_full,
    X=X,
    Y=Y,
    ko_to_superclass=ko_to_superclass,
    out_path=sweep_out,
    cv_cfg=cv_cfg,
    embed_cfg=embed_cfg,
    model_cfg=model_cfg,
    prf_cfg=prf_cfg,
    metrics_cfg=metrics_cfg,
    sweep_cfg=cfg["fixed_param_sweep"],
    embed=embed,
)
sweep_df2.head()

,neigh_k,tau_mult,y_latent_k,metric_ridge,lam,min_prev_y_abs,y_detect_threshold,pseudocount_y,metric_max_pairs,tau_scale_k_nn,...,ood_lam_cap,ood_tau_inflate,ood_tau_gamma,use_metric_learning,outer_splits,seed,informed_splits,dm_union_strict,dm_union_raw,runtime_s
21,40,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.533911,0.464808,1.827661
16,38,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.533412,0.464842,1.842254
11,36,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.532314,0.464541,1.899043
6,34,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.531887,0.464073,2.038843
22,40,0.50,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.531655,0.445505,1.843363


In [32]:
sweep_df2.sort_values('dm_union_strict', ascending=False).head()

,neigh_k,tau_mult,y_latent_k,metric_ridge,lam,min_prev_y_abs,y_detect_threshold,pseudocount_y,metric_max_pairs,tau_scale_k_nn,...,ood_lam_cap,ood_tau_inflate,ood_tau_gamma,use_metric_learning,outer_splits,seed,informed_splits,dm_union_strict,dm_union_raw,runtime_s
21,40,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.533911,0.464808,1.827661
16,38,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.533412,0.464842,1.842254
11,36,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.532314,0.464541,1.899043
6,34,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.531887,0.464073,2.038843
22,40,0.50,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.531655,0.445505,1.843363


In [33]:
# Define a small fixed-parameter sweep (uses the same X_full/X/Y/ko_to_superclass as above), round 3: keep best ones, sweep y_latent_k
cfg["fixed_param_sweep"] = {
    "neigh_k": [38, 40, 42, 44],
    "tau_mult": [0.35],
    "y_latent_k": [0, 10, 20, 30],
    "metric_ridge": [1.0],
}

sweep_out3 = results_dir / "fixed_param_sweep3.tsv"
sweep_df3 = run_fixed_param_sweep_explicit(
    X_full=X_full,
    X=X,
    Y=Y,
    ko_to_superclass=ko_to_superclass,
    out_path=sweep_out,
    cv_cfg=cv_cfg,
    embed_cfg=embed_cfg,
    model_cfg=model_cfg,
    prf_cfg=prf_cfg,
    metrics_cfg=metrics_cfg,
    sweep_cfg=cfg["fixed_param_sweep"],
    embed=embed,
)
sweep_df3.head()

,neigh_k,tau_mult,y_latent_k,metric_ridge,lam,min_prev_y_abs,y_detect_threshold,pseudocount_y,metric_max_pairs,tau_scale_k_nn,...,ood_lam_cap,ood_tau_inflate,ood_tau_gamma,use_metric_learning,outer_splits,seed,informed_splits,dm_union_strict,dm_union_raw,runtime_s
13,44,0.35,10,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.534937,0.462399,1.848516
14,44,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.534759,0.465122,1.895034
12,44,0.35,0,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.534573,0.466863,1.935443
9,42,0.35,10,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.534531,0.462298,1.989625
10,42,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.534315,0.465052,2.022174


In [34]:
sweep_df3.sort_values('dm_union_strict', ascending=False).head()

,neigh_k,tau_mult,y_latent_k,metric_ridge,lam,min_prev_y_abs,y_detect_threshold,pseudocount_y,metric_max_pairs,tau_scale_k_nn,...,ood_lam_cap,ood_tau_inflate,ood_tau_gamma,use_metric_learning,outer_splits,seed,informed_splits,dm_union_strict,dm_union_raw,runtime_s
13,44,0.35,10,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.534937,0.462399,1.848516
14,44,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.534759,0.465122,1.895034
12,44,0.35,0,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.534573,0.466863,1.935443
9,42,0.35,10,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.534531,0.462298,1.989625
10,42,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.534315,0.465052,2.022174


-> minimal, keep y_latent_k = 10

In [39]:
# Define a small fixed-parameter sweep (uses the same X_full/X/Y/ko_to_superclass as above), round 4: keep best one, sweep y_detect_threshold
cfg["fixed_param_sweep"] = {
    "neigh_k": [40, 42, 44],
    "tau_mult": [0.35],
    "y_latent_k": [10, 20],
    "metric_ridge": [1.0],
    "y_detect_threshold": [1000.0, 1500.0, 2500.0],
    "metric_max_pairs": [3000, 5000, 7000],
}

sweep_out4 = results_dir / "fixed_param_sweep4.tsv"
sweep_df4 = run_fixed_param_sweep_explicit(
    X_full=X_full,
    X=X,
    Y=Y,
    ko_to_superclass=ko_to_superclass,
    out_path=sweep_out,
    cv_cfg=cv_cfg,
    embed_cfg=embed_cfg,
    model_cfg=model_cfg,
    prf_cfg=prf_cfg,
    metrics_cfg=metrics_cfg,
    sweep_cfg=cfg["fixed_param_sweep"],
    embed=embed,
)
sweep_df4.head()

,neigh_k,tau_mult,y_latent_k,metric_ridge,lam,min_prev_y_abs,y_detect_threshold,pseudocount_y,metric_max_pairs,tau_scale_k_nn,...,ood_lam_cap,ood_tau_inflate,ood_tau_gamma,use_metric_learning,outer_splits,seed,informed_splits,dm_union_strict,dm_union_raw,runtime_s
38,44,0.35,10,1.0,0.0,1,1000.0,5.000000e-07,7000,10,...,0.5,False,1.0,True,5,0,False,0.534937,0.462399,1.869812
37,44,0.35,10,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.534937,0.462399,1.912466
36,44,0.35,10,1.0,0.0,1,1000.0,5.000000e-07,3000,10,...,0.5,False,1.0,True,5,0,False,0.534937,0.462399,1.965578
47,44,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,7000,10,...,0.5,False,1.0,True,5,0,False,0.534759,0.465122,1.923357
46,44,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.534759,0.465122,1.959658


In [40]:
sweep_df4.sort_values('dm_union_strict', ascending=False).head()

,neigh_k,tau_mult,y_latent_k,metric_ridge,lam,min_prev_y_abs,y_detect_threshold,pseudocount_y,metric_max_pairs,tau_scale_k_nn,...,ood_lam_cap,ood_tau_inflate,ood_tau_gamma,use_metric_learning,outer_splits,seed,informed_splits,dm_union_strict,dm_union_raw,runtime_s
38,44,0.35,10,1.0,0.0,1,1000.0,5.000000e-07,7000,10,...,0.5,False,1.0,True,5,0,False,0.534937,0.462399,1.869812
36,44,0.35,10,1.0,0.0,1,1000.0,5.000000e-07,3000,10,...,0.5,False,1.0,True,5,0,False,0.534937,0.462399,1.965578
37,44,0.35,10,1.0,0.0,1,1000.0,5.000000e-07,5000,10,...,0.5,False,1.0,True,5,0,False,0.534937,0.462399,1.912466
47,44,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,7000,10,...,0.5,False,1.0,True,5,0,False,0.534759,0.465122,1.923357
45,44,0.35,20,1.0,0.0,1,1000.0,5.000000e-07,3000,10,...,0.5,False,1.0,True,5,0,False,0.534759,0.465122,1.954096


In [46]:
detect_threshold = model_cfg['y_detect_threshold']
Y_prefiltered = Y.copy()
Y_prefiltered = Y_prefiltered.mask(Y_prefiltered < detect_threshold, 0.0)
Y_prefiltered = Y_prefiltered.mask(Y_prefiltered < 3000, 0.0)
# Y = Y_prefiltered

In [51]:
# Fit embedding on all samples (X_full)
embed = fit_x_embedding_svd_clr(
    X_full,
    min_prev_x_abs=40,
    pseudocount_x=float(embed_cfg['pseudocount_x']),
    n_components=int(embed_cfg['n_components']),
    seed=int(cv_cfg['seed']),
)

embed_path = results_dir / 'embed.joblib'
joblib.dump(embed, embed_path)
print('Saved embedding to:', embed_path)

Saved embedding to: /Users/meyeanni/Desktop/git_sourdough/Intelligrate/results/embed.joblib


In [52]:
# Define a small fixed-parameter sweep (uses the same X_full/X/Y/ko_to_superclass as above), round 4: keep best one, sweep y_detect_threshold
cfg["fixed_param_sweep"] = {
    "neigh_k": [32, 36, 44],
    "tau_mult": [0.1, 0.35, 0.5],
    "y_latent_k": [10],
    "metric_ridge": [1.0],
    "min_prev_y_abs": [0, 1, 5, 10],
    "y_detect_threshold": [0, 1000.0, 1500.0, 2500.0],
    "metric_max_pairs": [3000],
}

sweep_out5 = results_dir / "fixed_param_sweep4.tsv"
sweep_df5 = run_fixed_param_sweep_explicit(
    X_full=X_full,
    X=X,
    Y=Y,
    ko_to_superclass=ko_to_superclass,
    out_path=sweep_out,
    cv_cfg=cv_cfg,
    embed_cfg=embed_cfg,
    model_cfg=model_cfg,
    prf_cfg=prf_cfg,
    metrics_cfg=metrics_cfg,
    sweep_cfg=cfg["fixed_param_sweep"],
    embed=embed,
)
sweep_df5.head()

,neigh_k,tau_mult,y_latent_k,metric_ridge,lam,min_prev_y_abs,y_detect_threshold,pseudocount_y,metric_max_pairs,tau_scale_k_nn,...,ood_lam_cap,ood_tau_inflate,ood_tau_gamma,use_metric_learning,outer_splits,seed,informed_splits,dm_union_strict,dm_union_raw,runtime_s
133,44,0.50,10,1.0,0.0,1,1000.0,5.000000e-07,3000,10,...,0.5,False,1.0,True,5,0,False,0.520065,0.428368,1.874334
129,44,0.50,10,1.0,0.0,0,1000.0,5.000000e-07,3000,10,...,0.5,False,1.0,True,5,0,False,0.520065,0.428368,1.882316
81,36,0.50,10,1.0,0.0,0,1000.0,5.000000e-07,3000,10,...,0.5,False,1.0,True,5,0,False,0.512432,0.423120,1.841318
85,36,0.50,10,1.0,0.0,1,1000.0,5.000000e-07,3000,10,...,0.5,False,1.0,True,5,0,False,0.512432,0.423120,1.844732
113,44,0.35,10,1.0,0.0,0,1000.0,5.000000e-07,3000,10,...,0.5,False,1.0,True,5,0,False,0.511913,0.429375,1.872023


-> 3000 for y_detect_threshold stays the best!

## 2) Full fit (final model)
We fit the embedding on all k-mer samples (same as before, but now parameters are exposed in case we would still want to update/improve some based on previous grid scans), then fit a final model on paired samples.

To pick hyperparameters, we take the most common (mode) values from the CV folds.
You can also set them manually.


### option 1 to get the best params from fits without sweep:

In [ ]:

from intelligrate.extrapolate.full_fit import fit_final_model, save_model



# Choose final hyperparameters from folds (mode) -> the ones that were most often selected
def mode_or_first(series, default):
    if series is None or series.empty:
        return default
    return series.mode().iloc[0]

neigh_k = int(mode_or_first(folds.get('neigh_k'), model_cfg['neigh_k_grid'][0]))
tau_mult = float(mode_or_first(folds.get('tau_mult'), model_cfg['tau_mult_grid'][0]))
lam = float(mode_or_first(folds.get('lam'), model_cfg['lam_grid'][0]))
y_latent_k = int(mode_or_first(folds.get('y_latent_k'), model_cfg['y_latent_k_grid'][0]))
metric_ridge = float(mode_or_first(folds.get('metric_ridge'), model_cfg['metric_ridge_grid'][0]))

print('Selected hyperparameters:', neigh_k, tau_mult, lam, y_latent_k, metric_ridge)

### option 2 to get the best params from fixed param sweep:

In [ ]:
from intelligrate.extrapolate.full_fit import fit_final_model, save_model

#which sweep_df should be considered?
sweep_final = sweep_df4.copy()

# Choose final hyperparameters from folds (mode) -> the ones that were most often selected
def mode_or_first(series, default):
    if series is None or series.empty:
        return default
    return series.mode().iloc[0]

neigh_k = int(mode_or_first(sweep_final.get('neigh_k'), model_cfg['neigh_k_grid'][0]))
tau_mult = float(mode_or_first(sweep_final.get('tau_mult'), model_cfg['tau_mult_grid'][0]))
lam = float(mode_or_first(sweep_final.get('lam'), model_cfg['lam_grid'][0]))
y_latent_k = int(mode_or_first(sweep_final.get('y_latent_k'), model_cfg['y_latent_k_grid'][0]))
metric_ridge = float(mode_or_first(sweep_final.get('metric_ridge'), model_cfg['metric_ridge_grid'][0]))

print('Selected hyperparameters:', neigh_k, tau_mult, lam, y_latent_k, metric_ridge)

then, we fit the final model which we will then apply for extrapolation.

In [ ]:
model = fit_final_model(
    X_train=X,
    Y_train_tpm=Y,
    embed=embed,
    min_prev_y_abs=int(model_cfg['min_prev_y_abs']),
    y_detect_threshold=float(model_cfg['y_detect_threshold']),
    pseudocount_y=float(model_cfg['pseudocount_y']),
    neigh_k=neigh_k,
    tau_mult=tau_mult,
    lam=lam,
    y_latent_k=y_latent_k,
    use_metric_learning=bool(model_cfg['use_metric_learning']),
    metric_ridge=metric_ridge,
    metric_max_pairs=int(model_cfg['metric_max_pairs']),
    tau_scale_k_nn=int(model_cfg.get('tau_scale_k_nn', 10)),
    ood_shrink=bool(model_cfg.get('ood_shrink', False)),
    ood_lam_base=float(model_cfg.get('ood_lam_base', 0.1)),
    ood_lam_cap=float(model_cfg.get('ood_lam_cap', 0.8)),
    seed=int(cv_cfg['seed']),
)

model_path = save_model(model, results_dir / 'model.joblib')
print('Saved model to:', model_path)

## 3) Full predict (= extrapolate)
We predict for all samples, then (optionally) evaluate on the paired subset if truth is available.


In [ ]:
from intelligrate.extrapolate.full_predict import predict_final_model, evaluate_paired_subset

# Predict for all samples (deployment)
Yhat_clr, Yhat_tss, diag = predict_final_model(X_full, model)

# We will save only the leakage-free version after replacing paired rows.


## Leakage-free paired predictions (fixed-parameter OOF)
We re-predict paired samples using fixed hyperparameters, training only on other samples.
Set outer_splits = N to emulate leave-one-out.


In [ ]:
from intelligrate.extrapolate.cv_knn import fixed_param_oof_knn_on_embedding

oof_fixed_clr, oof_fixed_tss, oof_fixed_folds = fixed_param_oof_knn_on_embedding(
    X=X,
    Y_tpm=Y,
    embed=embed,
    ko_to_superclass=ko_to_superclass,
    outer_splits=5,  # set to len(X) for leave-one-out
    seed=int(cv_cfg['seed']),
    min_prev_y_abs=int(model_cfg['min_prev_y_abs']),
    y_detect_threshold=float(model_cfg['y_detect_threshold']),
    pseudocount_y=float(model_cfg['pseudocount_y']),
    neigh_k=int(neigh_k),
    tau_mult=float(tau_mult),
    lam=float(lam),
    y_latent_k=int(y_latent_k),
    use_metric_learning=bool(model_cfg['use_metric_learning']),
    metric_max_pairs=int(model_cfg['metric_max_pairs']),
    metric_ridge=float(metric_ridge),
    tau_scale_k_nn=int(model_cfg.get('tau_scale_k_nn', 10)),
    ood_shrink=bool(model_cfg.get('ood_shrink', False)),
    ood_lam_base=float(model_cfg.get('ood_lam_base', 0.1)),
    ood_lam_cap=float(model_cfg.get('ood_lam_cap', 0.8)),
    ood_tau_inflate=bool(model_cfg.get('ood_tau_inflate', False)),
    ood_tau_gamma=float(model_cfg.get('ood_tau_gamma', 1.0)),
    informed_splits=bool(cv_cfg.get('informed_splits', False)),
    informed_kmeans_on='X',
    prf_thresh=float(prf_cfg['prf_thresh']),
    prf_weight=str(prf_cfg['prf_weight']),
)

oof_fixed_clr.to_csv(results_dir / 'oof_fixed_clr.tsv', sep='	')
oof_fixed_tss.to_csv(results_dir / 'oof_fixed_tss.tsv', sep='	')
oof_fixed_folds.to_csv(results_dir / 'oof_fixed_folds.tsv', sep='	', index=False)

# Replace paired rows in full predictions with leakage-free OOF predictions
# pred_full.* uses full-fit predictions for unpaired samples, but paired rows are OOF (leakage-free).
Yhat_clr_full_oof = Yhat_clr.copy()
Yhat_tss_full_oof = Yhat_tss.copy()
Yhat_clr_full_oof.loc[X.index] = oof_fixed_clr.loc[X.index]
Yhat_tss_full_oof.loc[X.index] = oof_fixed_tss.loc[X.index]

pred_full_prefix = results_dir / 'pred_full'
Yhat_clr_full_oof.to_csv(pred_full_prefix.with_suffix('.clr.tsv'), sep='	')
Yhat_tss_full_oof.to_csv(pred_full_prefix.with_suffix('.tss.tsv'), sep='	')

# Evaluate leakage-free paired predictions
metrics_oof = evaluate_paired_subset(
    truth_tpm=Y,
    pred_tss=oof_fixed_tss,
    pseudocount=float(model_cfg['pseudocount_y']),
    detect_threshold=float(model_cfg['y_detect_threshold']),
    prf_thresh=float(prf_cfg['prf_thresh']),
    prf_weight=str(prf_cfg['prf_weight']),
    compute_wclr=bool(metrics_cfg['compute_wclr']),
    compute_jsd=bool(metrics_cfg['compute_jsd']),
    compute_pathway=bool(metrics_cfg['compute_pathway_rmse']),
    compute_per_pathway=bool(metrics_cfg['pathway_rmse_per_group']),
    ko_to_group=ko_to_superclass,
    log1p_pathway=bool(metrics_cfg['pathway_rmse_log1p']),
)

pd.DataFrame([metrics_oof]).to_csv(results_dir / 'pred_paired_oof.metrics.tsv', sep='	', index=False)
metrics_oof


## Compare against PICRUSt2 + raw (non-union) metrics
We compute the same metrics for PICRUSt2 and compare union vs. raw (intersection-only) results.


 - dm_union_raw / dm_union_strict / dm_union_strict come from dm_spearman_union(...) with different fillna_zero and detect_threshold options:
      - raw = detect_threshold=0.0, fillna_zero=True
      - strict = detect_threshold=your threshold, fillna_zero=True
      - dm_union_strict  = detect_threshold=your threshold, fillna_zero=False
  - dm_spearman comes from evaluate_union_metrics(...).
    That function uses union + fillna_zero=True + detect_threshold, i.e. it is effectively the same as dm_union_strict.

  So:

  dm_spearman  ==  dm_union_strict     (should match)
  dm_union_strict     !=  dm_union_strict     (because fillna_zero=False)

In [ ]:
# Build a comparison table: intersection vs union_raw vs union_strict
metrics_spec = [
    {"name": "dm_spearman", "intersection": "intersection_dm_spearman", "union_raw": "dm_union_raw", "union_strict": "dm_union_strict"},
    {"name": "bray_spearman", "intersection": "intersection_bray_spearman", "union_raw": "bray_union_raw", "union_strict": "bray_union_strict"},
    {"name": "procrustes_aitchison_strict", "intersection": "intersection_procrustes_aitchison", "union_raw": "procrustes_aitchison_raw", "union_strict": "procrustes_aitchison_strict"},
    {"name": "procrustes_bray_strict", "intersection": "intersection_procrustes_bray", "union_raw": "procrustes_bray_raw", "union_strict": "procrustes_bray_strict"},
    {"name": "soft_precision", "intersection": "intersection_soft_precision", "union_raw": None, "union_strict": "soft_precision"},
    {"name": "soft_recall", "intersection": "intersection_soft_recall", "union_raw": None, "union_strict": "soft_recall"},
    {"name": "soft_f1", "intersection": "intersection_soft_f1", "union_raw": None, "union_strict": "soft_f1"},
    {"name": "wclr_mse", "intersection": "intersection_wclr_mse", "union_raw": None, "union_strict": "wclr_mse"},
    {"name": "jsd", "intersection": "intersection_jsd", "union_raw": None, "union_strict": "jsd"},
    {"name": "pathway_rmse", "intersection": "intersection_pathway_rmse", "union_raw": None, "union_strict": "pathway_rmse"},
]


def _get(d, key):
    if d is None or key is None:
        return None
    return d.get(key)

rows = []
for spec in metrics_spec:
    name = spec["name"]
    model_inter = _get(metrics_oof, spec["intersection"])
    model_raw = _get(metrics_oof, spec["union_raw"])
    model_strict = _get(metrics_oof, spec["union_strict"])
    pic_inter = _get(picrust_metrics, spec["intersection"])
    pic_raw = _get(picrust_metrics, spec["union_raw"])
    pic_strict = _get(picrust_metrics, spec["union_strict"])
    rows.append({
        "metric": name,
        "model_intersection": model_inter,
        "model_union_raw": model_raw,
        "model_union_strict": model_strict,
        "picrust_intersection": pic_inter,
        "picrust_union_raw": pic_raw,
        "picrust_union_strict": pic_strict,
        "delta_intersection": (None if model_inter is None or pic_inter is None else model_inter - pic_inter),
        "delta_union_raw": (None if model_raw is None or pic_raw is None else model_raw - pic_raw),
        "delta_union_strict": (None if model_strict is None or pic_strict is None else model_strict - pic_strict),
    })

compare_table = pd.DataFrame(rows)
compare_table


Metric definitions (higher is better unless noted):

  - dm_spearman: Spearman correlation between upper triangles of Aitchison distance matrices
    (CLR space). Measures how well sample–sample relationships are preserved.

  - bray_spearman: Spearman correlation between upper triangles of Bray–Curtis distance matrices
    (TSS space). Also measures sample–sample structure, but in Bray–Curtis space.

  - procrustes_aitchison_strict: Procrustes similarity between ordinations of Aitchison distance matrices
    (CLR space). Higher means closer geometric alignment of the two ordinations.

  - procrustes_bray_strict: Procrustes similarity between ordinations of Bray–Curtis distance matrices
    (TSS space). Higher means closer geometric alignment.

  - soft_precision / soft_recall / soft_f1: Thresholded precision/recall/F1 on KO presence,
    computed in TSS space using the chosen threshold and weighting scheme.
    - soft_precision = TP / (TP + FP), meaning of all predicted present KOs, how many are truly present.
    - soft_recall = TP / (TP + FN), meaning of all truly present KOs, how many are predicted present.
    - soft_f1 = 2 * (soft_precision * soft_recall) / (soft_precision + soft_recall) (harmonic mean of precision and recall).

  - wclr_mse: Weighted MSE in CLR space (lower is better). Weights emphasize more stable features.

  - jsd: Jensen–Shannon divergence between TSS profiles (lower is better).

  - pathway_rmse: RMSE after aggregating KOs to pathways/superclasses (lower is better).

In [ ]:
# Visualize deltas (positive = model better than PICRUSt2 for correlation metrics)
#multiply all except wclr_mse by 100 (all values in all rows except the row 'wclr_mse'):
compare_table_for_plotting = compare_table.copy()
compare_table_for_plotting.loc[compare_table_for_plotting['metric'] != 'wclr_mse', ['delta_raw', 'delta_union']] *= 100
        
if picrust_metrics is not None:
    plot_df = compare_table_for_plotting.set_index('metric')[['delta_raw', 'delta_union']]
    plot_df.plot(kind='bar', figsize=(8, 4))
    plt.axhline(0, color='black', linewidth=1)
    plt.title('Model - PICRUSt2 (raw vs union)')
    plt.ylabel('Delta')
    plt.tight_layout()
    plt.show()


## Procrustes overlays with ellipses
We overlay ordinations for truth vs prediction and truth vs PICRUSt2 (if available).


In [ ]:
from matplotlib.patches import Ellipse
from scipy.spatial import procrustes
from sklearn.manifold import MDS
#import pdist, squareform
from scipy.spatial.distance import pdist, squareform

def _mds_coords_from_dm(D, random_state=42):
    mds = MDS(n_components=2, dissimilarity='precomputed', random_state=random_state)
    return mds.fit_transform(D)

def _plot_ellipse(ax, pts, color):
    cov = np.cov(pts[:, :2].T)
    mean = pts[:, :2].mean(axis=0)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(vals)
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle, edgecolor=color, facecolor='none', lw=3)
    ax.add_patch(ellipse)

def clr_rows(df: pd.DataFrame, pseudocount: float = 0.5) -> pd.DataFrame:
    X = df.to_numpy(dtype=float)
    X = X + float(pseudocount)
    X = np.log(X)
    X = X - X.mean(axis=1, keepdims=True)
    return pd.DataFrame(X, index=df.index, columns=df.columns)

def aitchison_dm(Y_clr: pd.DataFrame) -> np.ndarray:
    # Euclidean in CLR space equals Aitchison distance
    return squareform(pdist(Y_clr.to_numpy(dtype=float), metric="euclidean"))

def plot_procrustes_overlay(truth_tss, pred_tss, title):
    # align by intersection of KOs
    common = truth_tss.columns.intersection(pred_tss.columns)
    t = truth_tss.loc[:, common]
    p = pred_tss.loc[:, common]
    t = t.div(t.sum(axis=1).replace(0, np.nan), axis=0)
    p = p.div(p.sum(axis=1).replace(0, np.nan), axis=0)

    t_clr = clr_rows(t, pseudocount=float(model_cfg['pseudocount_y']))
    p_clr = clr_rows(p, pseudocount=float(model_cfg['pseudocount_y']))
    good = t_clr.notna().all(axis=1) & p_clr.notna().all(axis=1)
    t_clr = t_clr.loc[good]
    p_clr = p_clr.loc[good]

    D_t = aitchison_dm(t_clr)
    D_p = aitchison_dm(p_clr)

    coords_t = _mds_coords_from_dm(D_t)
    coords_p = _mds_coords_from_dm(D_p)

    mt, mp, disparity = procrustes(coords_t, coords_p)

    fig, ax = plt.subplots(figsize=(4.2, 3.7))
    ax.scatter(mt[:, 0], mt[:, 1], s=25, facecolors='none', edgecolors='#345084', label='Truth')
    ax.scatter(mp[:, 0], mp[:, 1], s=25, facecolors='none', edgecolors='#CB6BCE', label='Prediction')
    _plot_ellipse(ax, mt, '#345084')
    _plot_ellipse(ax, mp, '#CB6BCE')
    ax.set_title(f'{title} (disparity={disparity:.3f})')
    ax.set_xlabel('MDS1')
    ax.set_ylabel('MDS2')
    ax.legend()
    plt.tight_layout()
    plt.show()

# Truth vs model
plot_procrustes_overlay(Y, oof_fixed_tss, title='Truth vs model')

# Truth vs PICRUSt2 (if available)
if 'picrust' in locals() and picrust is not None:
    plot_procrustes_overlay(Y, picrust, title='Truth vs PICRUSt2')


## Pathway deviation plots (log10)
We compare pathway-level deviations vs. truth for model and PICRUSt2 (raw vs union).


In [ ]:
truth_pw

In [ ]:
model_pw

In [ ]:
pic_pw = None
if 'picrust' in locals() and picrust is not None:
    pic_pw = _aggregate_to_pathway(picrust, ko_to_superclass)

pic_pw    

In [ ]:
def _aggregate_to_pathway(tss, ko_to_group):
    common = [c for c in tss.columns if c in ko_to_group]
    if not common:
        return pd.DataFrame(index=tss.index)
    groups = [ko_to_group[c] for c in common]
    out = tss.loc[:, common].copy()
    out.columns = groups
    return out.groupby(level=0, axis=1).sum()

def _align_union(a, b):
    cols = a.columns.union(b.columns)
    return a.reindex(columns=cols, fill_value=0.0), b.reindex(columns=cols, fill_value=0.0)

def _align_intersection(a, b):
    cols = a.columns.intersection(b.columns)
    return a.loc[:, cols], b.loc[:, cols]

def build_deviation_long(tables_aligned, ref_name='truth', eps=1e-6):
    ref = tables_aligned[ref_name]
    rows = []
    for cfg, df in tables_aligned.items():
        if cfg == ref_name:
            continue
        common = df.index.intersection(ref.index)
        A = df.loc[common]
        R = ref.loc[common]
        dev = np.log10(A + eps) - np.log10(R + eps)
        long = dev.stack().reset_index()
        long.columns = ['sample', 'pathway', 'deviation_log10']
        long['cfg'] = cfg
        rows.append(long)
    return pd.concat(rows, ignore_index=True)

# Aggregate to pathways
truth_pw = _aggregate_to_pathway(Y.div(Y.sum(axis=1).replace(0, np.nan), axis=0), ko_to_superclass)
model_pw = _aggregate_to_pathway(oof_fixed_tss, ko_to_superclass)

pic_pw = None
if 'picrust' in locals() and picrust is not None:
    pic_pw = _aggregate_to_pathway(picrust, ko_to_superclass)
    #make relative abundance of pic_pw:
    pic_pw = pic_pw.div(pic_pw.sum(axis=1).replace(0, np.nan), axis=0)

# Build raw (intersection) and union-aligned tables
tables_raw = {'truth': truth_pw}
t_m, m_m = _align_intersection(truth_pw, model_pw)
tables_raw['model_raw'] = m_m
if pic_pw is not None:
    t_p, p_p = _align_intersection(truth_pw, pic_pw)
    tables_raw['picrust_raw'] = p_p

tables_union = {'truth': truth_pw}
t_u, m_u = _align_union(truth_pw, model_pw)
tables_union['model_union'] = m_u
if pic_pw is not None:
    t_u2, p_u2 = _align_union(truth_pw, pic_pw)
    tables_union['picrust_union'] = p_u2

dev_raw = build_deviation_long(tables_raw, ref_name='truth', eps=1e-6)
dev_union = build_deviation_long(tables_union, ref_name='truth', eps=1e-6)

# Plot distributions (box + strip)
import seaborn as sns

def _plot_dev(dev_long, title):
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=dev_long, x='cfg', y='deviation_log10', showfliers=False)
    sns.stripplot(data=dev_long, x='cfg', y='deviation_log10', color='black', alpha=0.2, jitter=0.2, size=2)
    plt.axhline(0, color='black', lw=1)
    plt.title(title)
    plt.xlabel('')
    plt.ylabel('log10 deviation vs truth')
    plt.xticks(rotation=25, ha='right')
    plt.tight_layout()
    plt.show()

_plot_dev(dev_raw, 'Pathway deviations (raw/intersection)')
_plot_dev(dev_union, 'Pathway deviations (union)')


## Pathway statistics (Wilcoxon + CLD)
We run paired Wilcoxon tests on per-sample pathway deviations and visualize CLD letters.


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import seaborn as sns
import matplotlib.pyplot as plt

def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    p = np.asarray(pvals, float)
    out = np.full_like(p, np.nan)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return out
    pv = p[ok]
    order = np.argsort(pv)
    ranked = pv[order]
    m = len(ranked)
    q = ranked * m / (np.arange(1, m + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    tmp = np.empty_like(pv)
    tmp[order] = np.clip(q, 0, 1)
    out[ok] = tmp
    return out

def pairwise_paired_wilcoxon_for_pathway(dev_long: pd.DataFrame, pathway: str, cfg_order=None,
                                         min_pairs=10, zero_method='wilcox',
                                         alternative='two-sided') -> pd.DataFrame:
    d = dev_long[dev_long['pathway'] == pathway].copy()
    if d.empty:
        return pd.DataFrame(columns=['pathway','cfg1','cfg2','p','p_adj','n_pairs'])
    if cfg_order is None:
        cfgs = sorted(d['cfg'].unique())
    else:
        cfgs = [c for c in cfg_order if c in d['cfg'].unique()]
    by = {cfg: d[d['cfg'] == cfg].set_index('sample')['deviation_log10'] for cfg in cfgs}
    rows = []
    for i in range(len(cfgs)):
        for j in range(i + 1, len(cfgs)):
            a, b = cfgs[i], cfgs[j]
            sa, sb = by[a], by[b]
            common = sa.index.intersection(sb.index)
            if len(common) < min_pairs:
                rows.append({'pathway': pathway, 'cfg1': a, 'cfg2': b, 'p': np.nan, 'n_pairs': len(common)})
                continue
            diff = (sa.loc[common] - sb.loc[common]).to_numpy(dtype=float)
            diff = diff[np.isfinite(diff)]
            if diff.size < min_pairs:
                rows.append({'pathway': pathway, 'cfg1': a, 'cfg2': b, 'p': np.nan, 'n_pairs': int(diff.size)})
                continue
            if np.allclose(diff, 0):
                p = 1.0
            else:
                p = float(wilcoxon(diff, zero_method=zero_method, alternative=alternative).pvalue)
            rows.append({'pathway': pathway, 'cfg1': a, 'cfg2': b, 'p': p, 'n_pairs': int(diff.size)})
    out = pd.DataFrame(rows)
    out['p_adj'] = bh_fdr(out['p'].to_numpy())
    return out

def wilcoxon_vs_zero_for_pathway(dev_long: pd.DataFrame, pathway: str, cfg_order=None,
                                 min_n=10, zero_method='wilcox',
                                 alternative='two-sided') -> pd.DataFrame:
    d = dev_long[dev_long['pathway'] == pathway].copy()
    if d.empty:
        return pd.DataFrame(columns=['pathway','cfg','p','p_adj','n','median'])
    if cfg_order is None:
        cfgs = sorted(d['cfg'].unique())
    else:
        cfgs = [c for c in cfg_order if c in d['cfg'].unique()]
    rows = []
    for cfg in cfgs:
        x = d.loc[d['cfg'] == cfg, 'deviation_log10'].to_numpy(dtype=float)
        x = x[np.isfinite(x)]
        n = int(x.size)
        if n < min_n:
            rows.append({'pathway': pathway, 'cfg': cfg, 'p': np.nan, 'n': n, 'median': np.nan})
            continue
        med = float(np.median(x))
        if np.allclose(x, 0):
            p = 1.0
        else:
            p = float(wilcoxon(x, zero_method=zero_method, alternative=alternative).pvalue)
        rows.append({'pathway': pathway, 'cfg': cfg, 'p': p, 'n': n, 'median': med})
    out = pd.DataFrame(rows)
    out['p_adj'] = bh_fdr(out['p'].to_numpy())
    return out

def compute_cld_letters(cfgs, pairwise_df, p_col='p_adj', alpha=0.05, value_by_cfg=None):
    p_lookup = {}
    for _, r in pairwise_df.iterrows():
        a, b = r['cfg1'], r['cfg2']
        p_lookup[tuple(sorted((a, b)))] = r[p_col]
    def nonsig(a, b):
        if a == b:
            return True
        p = p_lookup.get(tuple(sorted((a, b))), np.nan)
        if not np.isfinite(p):
            return False
        return p >= alpha
    if value_by_cfg is not None:
        order = list(value_by_cfg.loc[cfgs].sort_values(ascending=False).index)
    else:
        order = list(cfgs)
    letter_groups = []
    for cfg in order:
        placed = False
        for grp in letter_groups:
            if all(nonsig(cfg, other) for other in grp):
                grp.append(cfg)
                placed = True
                break
        if not placed:
            letter_groups.append([cfg])
    def idx_to_letters(i):
        s = ''
        i0 = i
        while True:
            s = chr(ord('a') + (i0 % 26)) + s
            i0 = i0 // 26 - 1
            if i0 < 0:
                break
        return s
    cld = {cfg: '' for cfg in cfgs}
    for gi, grp in enumerate(letter_groups):
        L = idx_to_letters(gi)
        for cfg in grp:
            cld[cfg] += L
    return cld

def plot_pathway_pairwise_cld(
    dev_long: pd.DataFrame,
    pathway: str,
    pairwise_df: pd.DataFrame,
    cfg_order=None,
    kind='box',
    palette=None,
    figsize=(6, 4),
    sig_cutoff=0.05,
    p_col='p_adj',
    cld_order_by='median',
    y_max_cap=None,
    savepath=None,
):
    d = dev_long[dev_long['pathway'] == pathway].copy()
    if d.empty:
        raise ValueError(f"No data for pathway '{pathway}'.")
    if cfg_order is None:
        cfgs = sorted(d['cfg'].unique())
    else:
        cfgs = [c for c in cfg_order if c in d['cfg'].unique()]
    if y_max_cap is not None:
        d = d[np.abs(d['deviation_log10']) <= y_max_cap]
    if palette is None:
        palette = sns.color_palette(n_colors=len(cfgs))
    fig, ax = plt.subplots(figsize=figsize)
    if kind == 'violin':
        sns.violinplot(data=d, x='cfg', y='deviation_log10',
            order=cfgs, palette=palette, inner=None, cut=0, linewidth=1, ax=ax)
    else:
        sns.boxplot(data=d, x='cfg', y='deviation_log10',
            order=cfgs, palette=palette, showfliers=False, linewidth=2, ax=ax,
            boxprops=dict(alpha=0.5))
    sns.stripplot(data=d, x='cfg', y='deviation_log10',
        order=cfgs, color='#751C6DFF', alpha=0.35, jitter=0.25, size=2.5, ax=ax)
    ax.axhline(0, lw=1, color='black')
    ax.set_title(f"{pathway} — pairwise Wilcoxon", fontsize=16, loc='left')
    ax.set_xlabel('')
    ax.set_ylabel('log10 deviation vs shotgun', fontsize=14)
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=14)
    plt.xticks(rotation=45, ha='right')
    sns.despine(ax=ax, top=True, right=True)
    if cld_order_by == 'median':
        stat = d.groupby('cfg')['deviation_log10'].median()
    elif cld_order_by == 'mean':
        stat = d.groupby('cfg')['deviation_log10'].mean()
    else:
        stat = None
    cld = compute_cld_letters(cfgs, pairwise_df, p_col=p_col, alpha=sig_cutoff, value_by_cfg=stat)
    group_max = d.groupby('cfg')['deviation_log10'].max()
    y_range = (d['deviation_log10'].max() - d['deviation_log10'].min()) if len(d) else 1.0
    offset = max(0.05, 0.05 * y_range)
    for xi, cfg in enumerate(cfgs):
        if cfg not in group_max or pd.isna(group_max[cfg]):
            continue
        ax.text(xi, float(group_max[cfg]) + offset, cld.get(cfg, ''),
            ha='center', va='bottom', fontsize=14, color='black')
    cur_ymin, cur_ymax = ax.get_ylim()
    ax.set_ylim(cur_ymin, max(cur_ymax, float(group_max.max()) + 2 * offset))
    if y_max_cap is not None:
        ax.set_ylim(-y_max_cap, y_max_cap)
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=300, bbox_inches='tight')
    plt.show()
    return fig

def p_to_stars(p):
    if not np.isfinite(p):
        return ''
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return 'ns'

def plot_pathway_vs_zero(
    dev_long: pd.DataFrame,
    pathway: str,
    vs0_df: pd.DataFrame,
    cfg_order=None,
    kind='box',
    palette=None,
    figsize=(6, 4),
    sig_cutoff=0.05,
    p_col='p_adj',
    label_style='stars',
    y_max_cap=None,
    savepath=None,
):
    d = dev_long[dev_long['pathway'] == pathway].copy()
    if d.empty:
        raise ValueError(f"No data for pathway '{pathway}'.")
    if cfg_order is None:
        cfgs = sorted(d['cfg'].unique())
    else:
        cfgs = [c for c in cfg_order if c in d['cfg'].unique()]
    if y_max_cap is not None:
        d = d[np.abs(d['deviation_log10']) <= y_max_cap]
    if palette is None:
        palette = sns.color_palette(n_colors=len(cfgs))
    fig, ax = plt.subplots(figsize=figsize)
    if kind == 'violin':
        sns.violinplot(data=d, x='cfg', y='deviation_log10',
            order=cfgs, palette=palette, inner=None, cut=0, linewidth=1, ax=ax)
    else:
        sns.boxplot(data=d, x='cfg', y='deviation_log10',
            order=cfgs, palette=palette, showfliers=False, linewidth=2, ax=ax,
            boxprops=dict(alpha=0.5))
    sns.stripplot(data=d, x='cfg', y='deviation_log10',
        order=cfgs, color='#751C6DFF', alpha=0.35, jitter=0.25, size=2.5, ax=ax)
    ax.axhline(0, lw=1, color='black')
    ax.set_title(f"{pathway} — Wilcoxon vs 0", fontsize=16, loc='left')
    ax.set_xlabel('')
    ax.set_ylabel('log10 deviation vs shotgun', fontsize=14)
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=14)
    plt.xticks(rotation=45, ha='right')
    sns.despine(ax=ax, top=True, right=True)
    group_max = d.groupby('cfg')['deviation_log10'].max()
    y_range = (d['deviation_log10'].max() - d['deviation_log10'].min()) if len(d) else 1.0
    offset = max(0.05, 0.05 * y_range)
    vs0_df = vs0_df.set_index('cfg')
    for xi, cfg in enumerate(cfgs):
        if cfg not in group_max or pd.isna(group_max[cfg]):
            continue
        q = float(vs0_df.loc[cfg, p_col]) if cfg in vs0_df.index else np.nan
        if label_style == 'q':
            lab = f"q={q:.3g}" if np.isfinite(q) else ''
        else:
            lab = p_to_stars(q)
        ax.text(xi, float(group_max[cfg]) + offset, lab,
            ha='center', va='bottom', fontsize=12, color='black')
    cur_ymin, cur_ymax = ax.get_ylim()
    ax.set_ylim(cur_ymin, max(cur_ymax, float(group_max.max()) + 2 * offset))
    if y_max_cap is not None:
        ax.set_ylim(-y_max_cap, y_max_cap)
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=300, bbox_inches='tight')
    plt.show()
    return fig

def plot_two_per_pathway(
    dev_long: pd.DataFrame,
    pathway: str,
    cfg_order=None,
    kind='box',
    palette=None,
    min_pairs=10,
    min_n=10,
    sig_cutoff=0.05,
    y_max_cap=None,
    outdir='results/pathway_stats',
):
    import os
    os.makedirs(outdir, exist_ok=True)
    pairwise_df = pairwise_paired_wilcoxon_for_pathway(
        dev_long, pathway, cfg_order=cfg_order, min_pairs=min_pairs
    )
    vs0_df = wilcoxon_vs_zero_for_pathway(
        dev_long, pathway, cfg_order=cfg_order, min_n=min_n
    )
    safe = pathway.replace('/', '_').replace(' ', '_')
    plot_pathway_pairwise_cld(
        dev_long, pathway, pairwise_df,
        cfg_order=cfg_order, kind=kind, palette=palette,
        sig_cutoff=sig_cutoff, y_max_cap=y_max_cap,
        savepath=f"{outdir}/{safe}__pairwise_cld.pdf"
    )
    plot_pathway_vs_zero(
        dev_long, pathway, vs0_df,
        cfg_order=cfg_order, kind=kind, palette=palette,
        sig_cutoff=sig_cutoff, y_max_cap=y_max_cap,
        label_style='stars',
        savepath=f"{outdir}/{safe}__vs0.pdf"
    )
    return pairwise_df, vs0_df


In [ ]:
dev_union

In [ ]:
# Example: run on a single pathway (most frequent)
#top_pathway = dev_union['pathway'].value_counts().index[15]
cfg_order = ['model_raw', 'model_union']
if 'picrust_union' in dev_union['cfg'].unique():
    cfg_order.extend(['picrust_raw', 'picrust_union'])

for pathway in dev_union['pathway'].unique():
    print(f"Processing pathway: {pathway}")
    pairwise_df, vs0_df = plot_two_per_pathway(
        dev_union,
        pathway=pathway,
        cfg_order=cfg_order,
        kind='box',
        palette=None,
        min_pairs=10,
        min_n=10,
        sig_cutoff=0.05,
        y_max_cap=None,
        outdir=str(results_dir / 'pathway_stats'),
    )
    pairwise_df.head()


In [ ]:
# Quick OOD diagnostic plot
plt.figure(figsize=(6, 3))
plt.hist(diag['ood_nn_min'], bins=30, color='#345084', alpha=0.8)
plt.title('OOD nearest-neighbor distance (pred_full)')
plt.xlabel('ood_nn_min')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


## Interpreting outputs
- `results/oof_clr.tsv`, `results/oof_tss.tsv`: out-of-fold predictions from nested CV
- `results/folds.tsv`: per-fold metrics + selected hyperparameters
- `results/summary.json` / `results/summary.tsv`: overall metrics
- `results/embed.joblib`: fitted embedding for k-mers
- `results/model.joblib`: final model artifact
- `results/pred_full.*`: predictions for all samples
- `results/pred_paired.metrics.tsv`: metrics when truth is provided
